# P12 業務に入れる ― 5 つの場面を正解と照合する

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

架空のソラハ電機の 5 つの業務 (規程の問い合わせ・経費と発注の登録・顧客台帳の照会・月次レポート・契約書の一次レビュー) を最後まで流し、根拠・登録された記録・監査ログ・レポートの数字・総合の判定を、正解と機械で照合します。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. 5 つの場面と正解を見る
場面ごとに、何を正解として照合するかが違います。規程の問い合わせの正解 7 問は、規程を読んで作りました。

In [ ]:
from usecases import UC_TITLE, all_cases
what = {1: "根拠の文書番号と節", 2: "登録された記録 (state/*.jsonl)", 3: "見えた項目・隠れた項目・監査ログ",
        4: "レポートの数字 13 個", 5: "総合の判定と当たる項目"}
for uc, title in UC_TITLE.items():
    cs = all_cases(uc)
    print(f"UC{uc} {title}: {len(cs)} 件 ({cs[0]['id']} ほか) / 照合するもの: {what[uc]}")
k = all_cases(1)[3]
print(f"例 {k['id']} {k['q']} → 正解 {k['doc']} {k['sections'][0]}")

## 2. 規程の問い合わせ (kb)
答えの最後の「根拠: 文書番号 節」が正解の節か、その節を道具で実際に見ていたか、を照合します。K-4 は検索の 1 位が改定履歴になる問いです。

In [ ]:
from run_usecases import amain as usecases
with safe():
    r = await usecases(["--uc", "1", "--only", "K-1,K-4", "--show"])

## 3. 経費・発注の登録 (expense・inventory)
登録の前に確認の画面が出ます。人の代わりの承認役は、画面の金額を依頼の伝票と見比べて決めます (PO-5 は予算を超えるので断る)。

In [ ]:
with safe():
    r = await usecases(["--uc", "2", "--only", "EX-2,EX-3,PO-5"])

## 4. 顧客台帳の照会と権限 (crm)
同じ問いを、担当営業 (U-01)・経理 (U-06)・情報システム (U-09) が聞きます。隠れた項目と監査ログを正解 (access_gold.json) と照合します。

In [ ]:
with safe():
    r = await usecases(["--uc", "3", "--only", "U-01/C-001,U-06/C-001,U-09/C-001"])

## 5. 月次レポート (report + Skill)
Skill の手順どおり、数字は道具から取り、計算は Skill のスクリプトで行います。レポートの数字 13 個を正解と照合します。

In [ ]:
with safe():
    r = await usecases(["--uc", "4", "--only", "2026-08", "--show"])

## 6. 契約書の一次レビュー (contract + Skill)
総合の判定は Skill のスクリプト (overall.py) が出します。判定と当たる項目を正解と照合します。

In [ ]:
with safe():
    r = await usecases(["--uc", "5", "--only", "LC-05,LC-07"])

## 7. 手を抜く筋書きで、採点が間違いを拾うか
検索の 1 位をそのまま引く・判定を確かめずに申請する・スクリプトを使わない、の 3 つ。採点の働きを見るため、ここだけ mock の筋書きで動かします (数字は Claude の性能の証拠にしない)。

In [ ]:
with safe():
    r = await usecases(["--uc", "1,2,5", "--only", "K-4,EX-3,LC-05", "--provider", "mock", "--plan", "sloppy"])

## 8. 監査ログを読む
顧客台帳の照会は、サーバの側で監査ログに残ります。だれが・どの顧客を・何を隠されたかを見ます。

In [ ]:
import json, os
with safe():
    r = await usecases(["--uc", "3", "--only", "U-05/C-001,U-04/IND-002"])
    p = os.path.join(r["state_dir"], "audit.jsonl")
    for e in [json.loads(l) for l in open(p, encoding="utf-8")]:
        print(e["at"], e["user"], e["tool"], e.get("customer"), "隠した:", e.get("hidden"))